<a href="https://colab.research.google.com/github/bhargavdhanisetti/AI_INSURANCE_CLAIM_AUTOMATION/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# W1 - NLP Text Preprocessing Pipeline
# ==========================================

# Install Required Libraries
# pip install pandas spacy scikit-learn fastapi uvicorn pytest

import pandas as pd
import spacy
import re

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Load Dataset
df = pd.read_csv("claim_data.csv")

# Display Dataset
print(df.head())

# ==========================================
# Create Sample Text Column
# ==========================================

# Combining useful columns into one text field
df["claim_text"] = (
    df["Diagnosis Code"].astype(str) + " " +
    df["Procedure Code"].astype(str) + " " +
    df["Reason Code"].astype(str)
)

# ==========================================
# Text Preprocessing Function
# ==========================================

def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Process using spaCy
    doc = nlp(text)

    # Remove stopwords
    tokens = [
        token.text
        for token in doc
        if not token.is_stop and not token.is_punct
    ]

    return " ".join(tokens)

# Apply preprocessing
df["clean_text"] = df["claim_text"].apply(preprocess_text)

print(df[["claim_text", "clean_text"]].head())

# ==========================================
# ICD Code Extraction
# ==========================================

def extract_icd_codes(text):

    pattern = r'\b[A-Z][0-9]{2}(?:\.[0-9]+)?\b'

    return re.findall(pattern, str(text))

df["icd_codes"] = df["Diagnosis Code"].apply(extract_icd_codes)

print(df[["Diagnosis Code", "icd_codes"]].head())

# ==========================================
# Save Cleaned Dataset
# ==========================================

df.to_csv("cleaned_claim_data.csv", index=False)

print("Preprocessing Completed Successfully ✅")

     Claim ID  Provider ID  Patient ID Date of Service  Billed Amount  \
0  0HO1FSN4AP    126528997  7936697103      08/07/2024            304   
1  9U86CI2P5A   6986719948  1547160031      06/21/2024            348   
2  1QEU1AIDAU   1355108115  2611585318      07/04/2024            235   
3  WH7XDS8CEO   9991055906  7167948632      05/26/2024            112   
4  M6OJEZ8KGI   7382167012  2140226267      07/16/2024            406   

   Procedure Code Diagnosis Code  Allowed Amount  Paid Amount Insurance Type  \
0           99231          A02.1             218          203       Self-Pay   
1           99213          A16.5             216          206       Medicare   
2           99213          A00.1             148          119     Commercial   
3           99215          A18.6              79           69       Medicare   
4           99238          A17.9             320          259       Medicare   

   Claim Status                    Reason Code Follow-up Required  \
0          